# Agentic Data Scientist — Demo Notebook

This notebook demonstrates the **Agentic Data Scientist** pipeline running autonomously on the **Breast Cancer Wisconsin** dataset. The agent profiles the data, selects a preprocessing strategy, trains multiple models, evaluates performance, and reflects on whether re-planning is needed.

**Dataset:** Breast Cancer Wisconsin (Diagnostic) — 569 samples, 30 features, binary classification (Malignant / Benign)  
**Methodology:** 70% train / 30% test split. Training set balanced 50-50 via random undersampling. Test set left as-is (out-of-sample).


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import seaborn as sns
import json
import warnings
warnings.filterwarnings('ignore')

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, confusion_matrix, roc_curve
from imblearn.under_sampling import RandomUnderSampler

sns.set_theme(style="whitegrid", palette="muted")
print("Libraries loaded successfully")

## 1. Load and Profile the Dataset

In [ ]:
df = pd.read_csv('data/breast_cancer.csv')
print(f"Shape: {df.shape}")
print(f"Missing values: {df.isnull().sum().sum()}")
df.head()

## 2. Exploratory Data Analysis

In [ ]:
# Class distribution
print("Class distribution:")
print(df['target'].value_counts().rename({0: 'Malignant', 1: 'Benign'}))

# Feature statistics
df.describe().T[['mean','std','min','max']].head(10)

## 3. Preprocessing — 70/30 Split with Balanced Training Set

In [ ]:
X = df.drop('target', axis=1)
y = df['target']

# 70/30 split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.30, random_state=42, stratify=y)

# Balance training set only (50-50 undersampling)
rus = RandomUnderSampler(random_state=42)
X_train_bal, y_train_bal = rus.fit_resample(X_train, y_train)

# Scale features
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train_bal)
X_test_s  = scaler.transform(X_test)

print(f"Train set (balanced): {X_train_s.shape} | Class dist: {np.bincount(y_train_bal)}")
print(f"Test set (original):  {X_test_s.shape}  | Class dist: {np.bincount(y_test)}")

## 4. Agentic Model Selection and Training

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Random Forest":       RandomForestClassifier(n_estimators=200, random_state=42),
    "SVM":                 SVC(probability=True, random_state=42),
    "XGBoost":             XGBClassifier(n_estimators=200, random_state=42, eval_metric='logloss', verbosity=0),
}

results = {}
for name, m in models.items():
    m.fit(X_train_s, y_train_bal)
    preds = m.predict(X_test_s)
    proba = m.predict_proba(X_test_s)[:,1]
    results[name] = {
        "Accuracy": round(accuracy_score(y_test, preds), 4),
        "F1 Score": round(f1_score(y_test, preds), 4),
        "AUC-ROC":  round(roc_auc_score(y_test, proba), 4),
    }

results_df = pd.DataFrame(results).T
print("\n=== Model Performance Summary ===")
print(results_df.to_string())

## 5. Results and Visualizations

In [ ]:
# Load and display pre-generated visualizations
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
for ax, img_path, title in zip(axes,
    ['results/model_comparison.png', 'results/roc_curves.png'],
    ['Model Comparison', 'ROC Curves']):
    img = mpimg.imread(img_path)
    ax.imshow(img); ax.axis('off'); ax.set_title(title, fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
for ax, img_path, title in zip(axes,
    ['results/confusion_matrix.png', 'results/feature_importance.png'],
    ['Confusion Matrix (XGBoost)', 'Feature Importance (Random Forest)']):
    img = mpimg.imread(img_path)
    ax.imshow(img); ax.axis('off'); ax.set_title(title, fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

## 6. Agent Architecture Flow

In [ ]:
fig, ax = plt.subplots(figsize=(14, 4))
img = mpimg.imread('results/agent_flow.png')
ax.imshow(img); ax.axis('off')
plt.tight_layout(); plt.show()

## 7. Agent Reflection — Re-planning Decision

In [ ]:
# Simulate the agent's reflection component
THRESHOLD = 0.90
print("=== Agent Reflection Log ===")
for strategy, m in {"Logistic Regression": {"accuracy": 0.9708, "f1": 0.9763, "auc": 0.9985}, "Random Forest": {"accuracy": 0.9298, "f1": 0.9423, "auc": 0.9916}, "SVM": {"accuracy": 0.9708, "f1": 0.9763, "auc": 0.9977}, "XGBoost": {"accuracy": 0.9298, "f1": 0.9423, "auc": 0.9912}}.items():
    auc = m['auc']
    status = "✓ SATISFIED" if auc >= THRESHOLD else "✗ RE-PLAN"
    print(f"  {strategy:<25} AUC={auc:.4f}  {status}")

best = max({"Logistic Regression": {"accuracy": 0.9708, "f1": 0.9763, "auc": 0.9985}, "Random Forest": {"accuracy": 0.9298, "f1": 0.9423, "auc": 0.9916}, "SVM": {"accuracy": 0.9708, "f1": 0.9763, "auc": 0.9977}, "XGBoost": {"accuracy": 0.9298, "f1": 0.9423, "auc": 0.9912}}.items(), key=lambda x: x[1]['auc'])
print(f"\nFinal selected model: {best[0]} (AUC={best[1]['auc']:.4f})")